In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


data = pd.read_excel("time_series_375_preprocess_en-2.xlsx")

In [ ]:
print(f"Number of rows in the training dataset: {len(data)}")
print(f"Number of features in the training dataset: {data.shape[1]}")

data.head()

In [ ]:
def fill_patient_id(df):
    current_patient_id = None
    for index, row in df.iterrows():
        if pd.notna(row['PATIENT_ID']):
            current_patient_id = row['PATIENT_ID']
        else:
            df.at[index, 'PATIENT_ID'] = current_patient_id

    df['PATIENT_ID'] = df['PATIENT_ID'].astype(int)

    return df


def get_last_measurement(df):
    last_measurement = df.groupby('PATIENT_ID').last().reset_index()
    return last_measurement


def calculate_hospital_stay_length(df):
    df['HOSPITAL_STAY_LENGTH'] = (df['Discharge time'] - df['Admission time']).dt.total_seconds() / 3600 # Convert to hours

    df.drop(columns=['Admission time', 'Discharge time'], inplace=True)

    return df    


def fill_missing_values(df):
    for column in df.columns:
        if df[column].dtype in ['float64', 'int64']:
            df[column] = df[column].fillna(df[column].mode()[0])
        else:
            df[column] = df[column].fillna(df[column].mode()[0])

    return df


def preprocess_data(df):
    df = fill_patient_id(df)
    df = get_last_measurement(df)
    df = calculate_hospital_stay_length(df)
    df = fill_missing_values(df)

    df.drop(columns=['RE_DATE', 'PATIENT_ID'], inplace=True)

    return df


data_preprocessed = preprocess_data(data)

X = data_preprocessed.drop(columns=["outcome"])
y = data_preprocessed["outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=1000)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

predictions = cross_val_predict(model, X, y, cv=cv)

print(classification_report(y, predictions))

In [ ]:
# --- Feature Importance via coefficients ---

model.fit(X_train, y_train)

feature_names = X.columns if hasattr(X, 'columns') else [f"feature_{i}" for i in range(X.shape[1])]

importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_[0],
    'abs_importance': np.abs(model.coef_[0])
}).sort_values('abs_importance', ascending=False)

# Top 10 features
top_10 = importance_df.head(10)
print("Top 10 features:")
print(top_10)

# Bottom 10 features
bottom_10 = importance_df.tail(10)
print("\nBottom 10 features:")
print(bottom_10)

# Optional: visualize top and bottom 10 together
plot_df = pd.concat([top_10, bottom_10]).sort_values('coefficient')

plot_df.plot(
    kind='barh', x='feature', y='coefficient',
    title='Top and Bottom Logistic Regression Feature Importance', legend=False
)
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

predictions = cross_val_predict(model, X, y, cv=cv)

print(classification_report(y, predictions))

In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(y, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix for Random Forest Classifier")
plt.show()

In [ ]:
model.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Top 10 features
top_10 = importance_df.head(10)
print("Top 10 features:")
print(top_10)


In [ ]:
# shap for random forest
import shap

model.fit(X_train, y_train)

X_shap = X_test.copy()
feature_names = X_shap.columns.tolist()

explainer = shap.TreeExplainer(model)
raw_shap = explainer(X_shap)


shap.plots.beeswarm(raw_shap[:,:,1])

In [ ]:
import matplotlib.pyplot as plt

values = [
    ("Lactate dehydrogenase", 0.130026, "#e63946"),
    ("neutrophils(%)", 0.093718, "#1b6ca8"),
    ("(%)lymphocyte", -0.091663, "#e63946"),
    ("Hypersensitive c-reactive protein", 0.071968, "#e63946"),
    ("D-D dimer", 0.049331, "#1b6ca8"),
    ("Prothrombin activity", -0.041475, "#1b6ca8")
]

# Sort by absolute value (largest first)
values_sorted = sorted(values, key=lambda x: abs(x[1]), reverse=True)

labels = [v[0] for v in values_sorted]
scores = [v[1] for v in values_sorted]
colors = [v[2] for v in values_sorted]

plt.figure()
bars = plt.barh(labels, scores)

# Apply colors
for bar, color in zip(bars, colors):
    bar.set_color(color)

max_abs = max(abs(s) for s in scores)
plt.xlim(-max_abs * 1.15, max_abs * 1.15)

# Put largest magnitude at the top
plt.gca().invert_yaxis()

plt.axvline(0, color='gray', linewidth=1)
plt.title("Most important features")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn import svm

model = svm.SVC(kernel='linear', random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

predictions = cross_val_predict(model, X, y, cv=cv)

print(classification_report(y, predictions))